# BLAST {#sec-blast}

*If you have a sequence, BLAST will the find any similar sequences in a large sequence database.*

In [ ]:
import vscodenb
import sandbox_widget

In [ ]:
#| echo: false
# This cell only defines the functions that draw the figures in this chapter.
# You do not need to read it (you will learn plotting in a later chapter).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

def _letter_row(ax, seq, y, colors=None):
    for x, letter in enumerate(seq):
        color = colors[x] if colors else INK
        ax.text(x + 0.5, y, letter, ha="center", va="center",
                family="monospace", fontsize=11, color=color, weight="bold")

def draw_words(ax, seq, w, rows):
    # rows is the number of word rows in the tallest panel, so that the
    # sequence and the words sit at the same heights in every panel
    n = len(seq) - w + 1
    _letter_row(ax, seq, rows + 1)
    for i in range(n):
        y = rows - i
        ax.add_patch(FancyBboxPatch((i + 0.08, y - 0.35), w - 0.16, 0.7,
                     boxstyle="round,pad=0,rounding_size=0.05",
                     facecolor='#999999', alpha=0.18, edgecolor='#999999', lw=1))
        _letter_row(ax, " " * i + seq[i:i+w], y)
    ax.set_xlim(0, len(seq))
    ax.set_ylim(0.3, rows + 1.7)
    # ax.set_title(f"w = {w}: {n} words", loc="left", fontsize=11)
    ax.axis("off")

def draw_word_figure(seq, sizes):
    rows = len(seq) - min(sizes) + 1
    fig, axes = plt.subplots(1, len(sizes), figsize=(4.2 * len(sizes), 0.3 * (rows + 1) + 0.6))
    for ax, w in zip(axes, sizes):
        draw_words(ax, seq, w, rows)
    plt.subplots_adjust(wspace=0.2)
    # plt.tight_layout()
    plt.show()

# def draw_word_figure(seq, size):
#     fig, ax = plt.subplots(1, 1, figsize=(5, 4))
#     draw_words(ax, seq, size, len(seq) - size + 1)
#     plt.tight_layout()
#     plt.show()

def draw_neighbor_figure(word, T, scores, alphabet, sizes, Ts):
    fig, (left, right) = plt.subplots(1, 2, figsize=(7, 3.6),
                                      gridspec_kw={"width_ratios": [1, 1.3]})
    # left: the word and the neighbors that reach T, grouped by score
    groups = {}
    for candidate in neighborhood(word, T, scores, alphabet):
        groups.setdefault(word_score(word, candidate, scores), []).append(candidate)
    y = 0
    for s in sorted(groups, reverse=True):
        left.text(-2.0, y, f"score {s}", ha="right", va="center", color=MUTED, fontsize=9)
        for k, candidate in enumerate(groups[s]):
            x0 = (k % 4) * (len(word) + 1)
            row = y - (k // 4) * 0.5
            for p, letter in enumerate(candidate):
                left.text(x0 + p, row, letter, family="monospace", fontsize=10, weight="bold",
                          ha="center", va="center", color=INK if letter == word[p] else ORANGE)
        y -= 0.8 * ((len(groups[s]) - 1) // 6 + 1) + 0.6
    left.set_xlim(-6, 6 * (len(word) + 1))
    left.set_ylim(y + 0.2, 0.6)
    left.axis("off")
    # left.set_title(f"Neighborhood of {word} at T = {T}: {len(neighborhood(word, T, scores, alphabet))} words",
    #                loc="left", fontsize=11)
    # right: neighborhood size as T changes, one line per word size
    colors = [BLUE, ORANGE, AQUA, YELLOW]
    for w, color in zip(sizes, colors):
        example = (word * w)[:w]
        counts = {}
        for candidate in all_words(w, alphabet):
            s = word_score(example, candidate, scores)
            counts[s] = counts.get(s, 0) + 1
        ys = []
        for t in Ts:
            total = 0
            for s in counts:
                if s >= t:
                    total = total + counts[s]
            ys.append(total)
        xs = [t for t, y in zip(Ts, ys) if y > 0]
        ys = [y for y in ys if y > 0]
        right.step(xs, ys, where="post", color=color, lw=2)
        right.text(Ts[0] - 0.4, ys[0], f"w = {w}", color=INK, ha="right", va="center", fontsize=9)
    right.set_yscale("log")
    right.set_xlabel("threshold T")
    right.set_ylabel("Neighborhood words")
    right.grid(axis="y", color=GRID, lw=0.8)
    right.set_xlim(Ts[0] - 3.5, Ts[-1] + 0.5)
    # right.set_title("Lower T, bigger neighborhood", loc="left", fontsize=11)
    plt.tight_layout()
    plt.show()

def draw_dotplots(query, subjects_and_hits, titles):
    fig, axes = plt.subplots(1, len(titles), figsize=(3.6 * len(titles), 3.4), squeeze=False)
    for ax, (subject, hits), title in zip(axes[0], subjects_and_hits, titles):
        xs = [j for i, j in hits]
        ys = [i for i, j in hits]
        ax.plot(xs, ys, "o", ms=3, color=BLUE, alpha=0.7)
        ax.set_xlim(0, len(subject))
        ax.set_ylim(len(query), 0)
        ax.set_xlabel("position in subject")
        ax.set_ylabel("position in query")
        ax.set_title(title, loc="left", fontsize=11)
    plt.tight_layout()
    plt.show()

def draw_tradeoff(sizes, random_hits, expected, found, divergences, n_relatives):
    fig, (left, right) = plt.subplots(1, 2, figsize=(7, 3.6))
    left.bar(sizes, random_hits, width=1.2, color=BLUE)
    left.plot(sizes, expected, "--", color=MUTED, lw=1.5)
    left.text(sizes[-2] + 0.4, expected[-2] * 3, "expected:\nm n / 4 ** w", color=MUTED, fontsize=9, va="bottom")
    left.set_yscale("symlog", linthresh=1)
    left.set_xticks(sizes)
    left.set_xlabel("word size w")
    left.set_ylabel("random hits to extend")
    left.grid(axis="y", color=GRID, lw=0.8)
    left.set_title("Speed: work wasted on random hits", loc="left", fontsize=11)
    for d, color in zip(divergences, [BLUE, ORANGE, AQUA]):
        identity = round(100 * (1 - 0.75 * d))
        right.plot(sizes, found[d], "-o", color=color, lw=2, ms=6)
        right.text(sizes[-1] + 0.3, found[d][-1], f"{identity}% identical", color=INK, va="center", fontsize=9)
    right.set_xticks(sizes)
    right.set_xlim(sizes[0] - 0.5, sizes[-1] + 3.5)
    right.set_ylim(-0.8, n_relatives + 0.8)
    right.set_yticks(range(0, n_relatives + 1, 5))
    right.set_xlabel("word size w")
    right.set_ylabel(f"relatives found (of {n_relatives})")
    right.grid(axis="y", color=GRID, lw=0.8)
    right.set_title("Sensitivity: relatives found", loc="left", fontsize=11)
    plt.tight_layout()
    plt.show()

## Why not just align against everything? {#sec-blast-why}

In @sec-biopython-align you met local alignment: set `mode = "local"` on a `PairwiseAligner`, and it finds the best-scoring stretch shared by two sequences. That algorithm is the Smith-Waterman algorithm ([Smith and Waterman, 1981](https://doi.org/10.1016/0022-2836(81)90087-5)), and it is guaranteed to find the best local alignment. So why not just align your sequence to every sequence in the database and keep the best ones?

The answer is arithmetic. Smith-Waterman fills in a table with one cell for every pair of positions, one from each sequence. Aligning a gene of 1,000 bases to another of 1,000 bases means filling in $1000 \times 1000 = 1,000,000$ cells. That is nothing for a computer. But a database is not one sequence. The human genome alone is about 3.1 billion bases long, and NCBI's nucleotide databases hold the genomes of a great many species besides.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Use Python as a calculator. How many cells must Smith-Waterman fill in to align a gene of 1,000 bases against the whole human genome? Suppose your computer can fill in a hundred million cells per second. How many hours does the search take? Decide roughly what you expect (seconds, minutes, hours?) before you compute it.

In [ ]:
%%sandbox
cells = 1000 * 3_100_000_000


Now picture an NCBI server answering thousands of such searches every hour, against databases that keep growing. Something has to give, and what BLAST gives up is the guarantee. BLAST is a *heuristic*: a shortcut that usually finds the best alignments, but is not promised to. In exchange it is very fast.

The shortcut rests on one observation about biology. If two sequences are related, the alignment between them almost always contains a few short stretches where the two sequences are identical or nearly so. So rather than aligning everything to everything, you first look for those short stretches, and you only spend time aligning where you find one. The earlier FASTA program ([Lipman and Pearson, 1985](https://doi.org/10.1126/science.2983426)) had already used this idea. BLAST, the *Basic Local Alignment Search Tool* ([Altschul et al., 1990](https://doi.org/10.1016/S0022-2836(05)80360-2)), made it faster and gave it a solid statistical footing. The paper describing it is one of the most cited scientific papers ever written.

## The original BLAST {#sec-blast-1990}

The original BLAST works in four steps:

1. Chop your sequence (the *query*) into short overlapping words, and list every word that scores well against one of them.
2. Scan the database for those words. Each occurrence is a *hit*.
3. Extend each hit in both directions into a longer alignment, and stop when the score starts falling.
4. Decide whether the alignments you found are better than what you would expect by chance.

You know enough Python to build a small version of each step yourself, so that is what we will do. Our version is slow and simple, and the real BLAST is fast and clever, but they do the same thing. We start with DNA, where everything is simplest, and turn to proteins in @sec-blast-protein-words.

### Step 1: overlapping words {#sec-blast-words}

Here is a short DNA query chopped into all its overlapping words of length `w`. The function slides a window of width `w` along the sequence, one base at a time. Predict how many words of length 4 there are before you run the cell.

In [ ]:
def words(seq, w):
    result = []
    for i in range(len(seq) - w + 1):
        result.append(seq[i:i+w])
    return result

query = "TGCATGCCAGTTACGGATCAAG"

print(len(query))
print(words(query, 4))
print(len(words(query, 4)))

Why `len(seq) - w + 1`? The last word must start `w - 1` bases before the end of the query, or it would be too short. The figure below shows the words of the first twelve bases for two word sizes. Each base belongs to up to `w` words, and every word overlaps the next one in all but one base.

In [ ]:
#| echo: false
#| label: fig-blast-words
#| fig-cap: "The overlapping words of the first twelve bases of the query, for word size 3 (left) and word size 6 (right). Longer words are fewer, and each one is a longer and rarer string."
draw_word_figure(query[:12], [3, 6])

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Write a formula for the number of words of length `w` in a sequence of length `L`. Use it to predict how many words of length 11 and of length 28 our query has, and how many words of length 11 a 1,000-base gene has. Then check the first two with `words`. What happens if `w` is longer than the query? Predict, then try.

In [ ]:
%%sandbox
def words(seq, w):
    result = []
    for i in range(len(seq) - w + 1):
        result.append(seq[i:i+w])
    return result

query = "TGCATGCCAGTTACGGATCAAG"


### Neighbor words {#sec-blast-neighbors}

Looking only for exact copies of the query's words is one option. But a word in a related sequence may differ from the query's word in a base or two, and BLAST lets you decide how different it may be. To do that, it needs a way to score two words against each other. For DNA we score each pair of bases: 2 if they match and -3 if they do not. These are the scores the standard `blastn` uses. We keep them in a dictionary of dictionaries, so `dna_scores["A"]["C"]` is the score of `A` against `C`:

In [ ]:
dna_scores = {}
for a in "ACGT":
    dna_scores[a] = {}
    for b in "ACGT":
        if a == b:
            dna_scores[a][b] = 2
        else:
            dna_scores[a][b] = -3

def word_score(word1, word2, scores):
    score = 0
    for k in range(len(word1)):
        score = score + scores[word1[k]][word2[k]]
    return score

print(word_score("ACGT", "ACGT", dna_scores))
print(word_score("ACGT", "AGGT", dna_scores))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

With these scores, a word of length `w` with `k` mismatches against another word scores $2(w - k) - 3k = 2w - 5k$. Do the substitution and reduction steps for `word_score("ACGT", "TCGA", dna_scores)` in your head, and check that the formula gives the same. Then run it.

In [ ]:
print(word_score("ACGT", "TCGA", dna_scores))

The *neighborhood* of a word is every word that scores at least some threshold $T$ against it. To find it, we can simply try every possible word of the same length. The function `all_words` builds them one letter at a time: first all words of length 1, then each of those with every letter added, and so on, `w` times.

In [ ]:
def all_words(w, alphabet):
    result = [""]
    for _ in range(w):
        longer = []
        for word in result:
            for letter in alphabet:
                longer.append(word + letter)
        result = longer
    return result

def neighborhood(word, T, scores, alphabet):
    neighbors = []
    for candidate in all_words(len(word), alphabet):
        if word_score(word, candidate, scores) >= T:
            neighbors.append(candidate)
    return neighbors

print(len(all_words(4, "ACGT")))
print(neighborhood("ACGT", 3, dna_scores, "ACGT"))

At $T = 3$, the neighborhood of `ACGT` holds `ACGT` itself and the twelve words that differ from it in exactly one base. The left side of the figure below shows them, with the changed base in orange. The right side shows how the size of the neighborhood depends on both settings. Lowering $T$ lets in words with more mismatches, and the neighborhood grows in steps, one step for each extra mismatch allowed. For the same $T$, a longer word has a bigger neighborhood, because there are more places for a mismatch to go.

In [ ]:
#| echo: false
#| label: fig-blast-neighbors
#| fig-cap: "Left: the neighborhood of ACGT at threshold T = 3, with each changed base shown in orange. Right: the number of words in the neighborhood of a DNA word as the threshold T changes, for word sizes 3 to 6 (note the logarithmic axis)."
draw_neighbor_figure("ACGT", 3, dna_scores, "ACGT", [3, 4, 5, 6], list(range(-12, 13)))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Use the formula $2w - 5k$ to predict the size of the neighborhood of `ACGT` at $T = 8$, at $T = 4$, and at $T = -2$. Hint: first work out how many mismatches each threshold allows, and then count the words with that many mismatches. Then check your three answers with `neighborhood`, and compare them with the line for `w = 4` in the figure.

In [ ]:
print(len(neighborhood("ACGT", 8, dna_scores, "ACGT")))

Here is the trade-off that runs through the rest of this chapter. A big neighborhood (small `w`, low $T$) means a related sequence is likely to contain at least one of the words, even if it has drifted far from the query. But it also means many more chance hits in unrelated sequences, and each of those takes time to check. A small neighborhood is fast but misses distant relatives. We measure this trade-off in @sec-blast-parameters.

### The lookup table {#sec-blast-lookup}

BLAST puts all the neighborhoods into one *lookup table*: for every word, it records the query positions it is a neighbor of. You know the perfect data structure for that, a dictionary from words to lists of positions:

In [ ]:
def build_lookup(query, w, T, scores, alphabet):
    lookup = {}
    for i in range(len(query) - w + 1):
        for word in neighborhood(query[i:i+w], T, scores, alphabet):
            if word not in lookup:
                lookup[word] = []
            lookup[word].append(i)
    return lookup

exact = build_lookup(query, 4, 8, dna_scores, "ACGT")
print(len(exact))
print(exact)

loose = build_lookup(query, 4, 3, dna_scores, "ACGT")
print(len(loose))

With $T = 8$, the top score for a word of length 4, each neighborhood holds only the word itself, so the table just records where each of the query's own words is: 19 words, each at its own position. With $T = 3$, each word brings its twelve one-mismatch neighbors along, and the table grows to many more words. The table is built once, from the query alone, before BLAST looks at the database at all.

### Step 2: scanning the database {#sec-blast-scan}

Now BLAST walks along each database sequence (the *subject*) one word at a time and looks each word up in the table. Looking up a key in a dictionary is very fast and takes about the same time no matter how big the dictionary is, which is why BLAST can afford to do it for every position in a database of billions of bases. Every time a word is found, BLAST notes a hit as a pair of positions: where the word is in the query, and where it is in the subject.

In [ ]:
def find_hits(lookup, subject, w):
    hits = []
    for j in range(len(subject) - w + 1):
        word = subject[j:j+w]
        if word in lookup:
            for i in lookup[word]:
                hits.append((i, j))
    return hits

subject = "CCGTATTGCATGCAAGTTACGCATCAAGTTCGA"
hits = find_hits(exact, subject, 4)
print(hits)

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Look at the hits as pairs `(i, j)`. For each of them, compute `j - i` in your head. What do you notice? Before you read on, explain what it means that most hits have the same value of `j - i`, and why a few do not. Hint: write the query above the subject on a piece of paper, shifted so the matching parts line up.

The difference `j - i` is called the *diagonal* of a hit. If you draw the query down one side of a grid and the subject along the other, as in the alignment table you know from Smith-Waterman, all hits with the same `j - i` lie on the same diagonal line through the grid. Hits on the same diagonal belong to the same ungapped alignment. Such a grid of hits is called a *dot plot*, and the figure below shows the one for our query and subject. The diagonal line of hits is where the subject contains a copy of the query with two changed bases. The gaps in the line are where those changes are.

In [ ]:
#| echo: false
#| label: fig-blast-dotplot
#| fig-cap: "A dot plot of the hits of the query in the subject, for word size 4 and exact matches. Each dot is a hit. The hits on one diagonal line mark the copy of the query in the subject."
draw_dotplots(query, [(subject, hits)], ["w = 4, T = 8"])

Keep the diagonals in mind. They are the key to the improved BLAST in @sec-blast-1997.

### Step 3: extending a hit {#sec-blast-extend}

A hit is only a few bases. To see whether it is part of something bigger, BLAST extends it, one base at a time, first to the right and then to the left, adding the score of each new pair of bases. The extension has no gaps: it just walks along the diagonal.

When should it stop? The score goes up when the bases agree and down when they do not, so a single mismatch should not end the extension. BLAST remembers the best score it has seen so far and stops when the score has dropped more than some amount $X$ below that best. Then it trims the alignment back to where the best score was. This is called the *X-drop* rule.

Read the function below slowly. The two `while` loops are the two directions. The variables `right` and `left` remember how far out the best score was found, so the function can return just that part.

In [ ]:
def extend(query, subject, i, j, w, scores, X):
    score = word_score(query[i:i+w], subject[j:j+w], scores)
    best = score

    # extend to the right
    right = w
    k = w
    while i + k < len(query) and j + k < len(subject) and score > best - X:
        score = score + scores[query[i+k]][subject[j+k]]
        k = k + 1
        if score > best:
            best = score
            right = k

    # extend to the left
    score = best
    left = 0
    k = 1
    while i - k >= 0 and j - k >= 0 and score > best - X:
        score = score + scores[query[i-k]][subject[j-k]]
        if score > best:
            best = score
            left = k
        k = k + 1

    return best, query[i-left:i+right], subject[j-left:j+right]

score, query_part, subject_part = extend(query, subject, 0, 6, 4, dna_scores, 10)
print(score)
print(query_part)
print(subject_part)

The result is a *high-scoring segment pair*, or HSP. That is the word you met in the Biopython BLAST results in @sec-biopython-blast, where every hit carried a list of HSPs.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Predict what happens to the HSP if you make $X$ small, say `3`, so that the extension gives up as soon as a single mismatch (which scores -3) has pulled the score 3 below the best. Will the HSP be shorter, longer, or the same? Write down your guess, then run `extend` with `X` set to `3` and to `10` and compare. Then try `4`. Which values let the extension reach past the mismatches, and why?

In [ ]:
print(extend(query, subject, 0, 6, 4, dna_scores, 10))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Extend every hit in `hits`, not just the first one, and print the score and the two segments for each. Before you run it, predict how many *different* HSPs you will get, and what the hits that are not on the main diagonal turn into. What does the answer tell you about how much work BLAST does for nothing?

In [ ]:
# extend every hit in hits and print the score and the two segments

## Word size, threshold, speed and sensitivity {#sec-blast-parameters}

A BLAST search spends most of its time extending hits, so the number of hits decides how fast it is. And it can only find a relative that contains at least one hit, so the number of hits in relatives decides how *sensitive* it is: how distant the relatives are that it can still find. Both depend on the word size and the threshold. Let us measure that.

First, how many chance hits should we expect? In random DNA, each base is one of four with equal chance, so a given word of length `w` matches a given position with probability $1/4^w$. A query with about $m$ words, scanned against a database with about $n$ positions, gives about

$$\frac{m \, n}{4^w}$$

random hits. Each extra base in the word divides that number by four.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Use the formula to predict the number of random hits for a query of 400 bases against a database of a million bases, for `w = 7` and `w = 11`. Then do the same for a 1,000-base gene against the 3.1 billion bases of the human genome with `w = 11` and with `w = 28`, the word size of `megablast`. Decide before you compute: roughly how many times fewer random hits does `w = 28` give than `w = 11`?

In [ ]:
%%sandbox
print(400 * 1_000_000 / 4 ** 7)


For DNA, the real BLAST does not use neighborhoods at all. It only looks for exact words, so the threshold is always the top score, and it controls the trade-off with the word size alone. That has a practical advantage, too: at the top score, the neighborhood of a word is just the word itself, so there is no need to try all $4^w$ possible words. That number is about 4 million for `w = 11` and more than a billion for `w = 15`, which would make our `build_lookup` very slow. Here is the shortcut:

In [ ]:
def exact_lookup(query, w):
    lookup = {}
    for i in range(len(query) - w + 1):
        word = query[i:i+w]
        if word not in lookup:
            lookup[word] = []
        lookup[word].append(i)
    return lookup

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

`exact_lookup(query, w)` should give the same table as `build_lookup` with $T = 2w$. That is a claim you can test. Predict whether the line below prints `True`, then run it. Then change it to test word size 5. What threshold must you give `build_lookup` then?

In [ ]:
print(exact_lookup(query, 4) == build_lookup(query, 4, 8, dna_scores, "ACGT"))

Now a test closer to a real search. We make a random query of 300 bases and a *relative* of it, in which about a third of the bases have been replaced by random ones. We hide the relative between two random stretches, and look at the dot plots for three word sizes. The function `random.choices` picks `k` random letters, and `"".join` glues them into a string. The call `random.seed(1)` makes Python pick the same random letters every time you run the cell, so you get the same results as this page.

In [ ]:
import random

random.seed(1)

def random_seq(n, alphabet):
    return "".join(random.choices(alphabet, k=n))

def mutate(seq, fraction, alphabet):
    new = []
    for letter in seq:
        if random.random() < fraction:
            new.append(random.choice(alphabet))
        else:
            new.append(letter)
    return "".join(new)

query_300 = random_seq(300, "ACGT")
relative = mutate(query_300, 0.35, "ACGT")
subject_900 = random_seq(300, "ACGT") + relative + random_seq(300, "ACGT")

plots = []
for w in [6, 11, 16]:
    plots.append((subject_900, find_hits(exact_lookup(query_300, w), subject_900, w)))

In [ ]:
#| echo: false
#| label: fig-blast-dotplots
#| fig-cap: "Dot plots of a random 300-base query against a 900-base subject that holds a relative of the query in its middle third, for three word sizes. With w = 6, the relative is buried in random hits. With w = 16, only a short piece of its diagonal is left."
draw_dotplots(query_300, plots, ["w = 6", "w = 11", "w = 16"])

With `w = 6`, the diagonal of the relative is there, but it is surrounded by random hits, and every one of them would be extended. With `w = 11`, the random hits are nearly gone and the diagonal still shows. With `w = 16`, the random hits are gone, but so is most of the diagonal: only one short piece of it survives, where the relative happens to share a long identical stretch with the query. One example is not much to go on, so let us count properly. The cell below builds a database of exactly one million bases, with 60 relatives of a 400-base query hidden in it: 20 each at three levels of divergence. For each word size, it counts the random hits, and how many of the 20 relatives at each level got at least one hit on their own diagonal. That is all BLAST needs to find a relative, because one hit is enough to start an extension. The dictionary `relative_starts` remembers where each relative starts, so we can recognize a hit on its diagonal: since the relatives have no gaps, that is a hit with `j - i` equal to the start.

In [ ]:
random.seed(3)

query_400 = random_seq(400, "ACGT")
divergences = [0.3, 0.45, 0.6]

pieces = []
relative_starts = {}
position = 0
for d in divergences:
    for _ in range(20):
        filler = random_seq(16_000, "ACGT")
        pieces.append(filler)
        position = position + len(filler)
        relative_starts[position] = d
        pieces.append(mutate(query_400, d, "ACGT"))
        position = position + len(query_400)
pieces.append(random_seq(16_000, "ACGT"))
database = "".join(pieces)

print(len(database))

Replacing a base by a random one leaves it unchanged one time in four, so a divergence of 0.3 leaves the relative about $1 - 0.75 \times 0.3 \approx 78\%$ identical to the query, 0.45 leaves it about 66% identical, and 0.6 about 55%. Now the experiment. The `time` module lets us measure how long the extensions take.

In [ ]:
import time

sizes = [7, 9, 11, 13, 15]
random_hits = []
expected = []
found = {}
for d in divergences:
    found[d] = []

for w in sizes:
    hits_w = find_hits(exact_lookup(query_400, w), database, w)

    start = time.perf_counter()
    for i, j in hits_w:
        extend(query_400, database, i, j, w, dna_scores, 20)
    seconds = time.perf_counter() - start

    relatives_hit = set()
    for i, j in hits_w:
        if j - i in relative_starts:
            relatives_hit.add(j - i)

    count = 0
    for i, j in hits_w:
        if j - i not in relative_starts:
            count = count + 1
    random_hits.append(count)
    expected.append(len(query_400) * len(database) / 4 ** w)

    for d in divergences:
        n = 0
        for s in relatives_hit:
            if relative_starts[s] == d:
                n = n + 1
        found[d].append(n)

    print(f"w = {w:2}: {len(hits_w):6} hits, {count:6} random, extending took {seconds:.2f} s,",
          "relatives found:", [found[d][-1] for d in divergences])

In [ ]:
#| echo: false
#| label: fig-blast-tradeoff
#| fig-cap: "Speed and sensitivity for exact words of different sizes, searching a 400-base query against a million random bases with 60 hidden relatives. Left: the number of random hits that must be extended, with the expected number as a dashed line. Right: how many of the 20 relatives at each level of identity got at least one hit."
draw_tradeoff(sizes, random_hits, expected, found, divergences, 20)

The two panels of the figure tell the whole story. Going from `w = 7` to `w = 15`, the number of random hits drops from tens of thousands to none, and the time spent extending drops with it. But the relatives disappear too, and the most distant ones disappear first. Nearly all of the 78% identical relatives are found at every word size. The 55% identical ones are all found with `w = 7`, but almost none with `w = 13` or more, because a sequence that differs from the query at almost every other base rarely contains a long unbroken identical stretch.

There is no setting that is best for everything. It depends on what you are looking for, and that is why NCBI offers several versions of nucleotide BLAST (@sec-blast-algorithms). `megablast` uses `w = 28` and is very fast, but only finds nearly identical sequences. The standard `blastn` uses `w = 11`. And you can set the word size yourself on the BLAST page, under *Algorithm parameters*.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Before you run anything, predict the number of random hits and the number of relatives found at each level for `w = 8` and `w = 10`. Use the formula for the random hits, and the figure to guess the relatives. Then add the two word sizes to `sizes` and run the experiment again to check.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Suppose you have a gene from a mouse and want to find the same gene in other mammals, where you expect the DNA to be about 85% identical. Then suppose you want to find it in fish, where you expect much less. For each case, which word size from the experiment would you choose, and why? Which number in the output would you look at to argue for your choice?

## Proteins: where neighbors earn their keep {#sec-blast-protein-words}

Proteins are different in two ways. First, there are 20 amino acids rather than four bases, so even a word of three letters is rare: a given word matches a random position only once in $20^3 = 8000$. Second, related proteins often have *similar* amino acids in the same place rather than identical ones, like a leucine (`L`) where the other has an isoleucine (`I`). Exact words would miss those, so protein BLAST uses short words of three letters and makes them sensitive with neighborhoods.

To score words, BLAST uses a substitution matrix, the same BLOSUM62 matrix ([Henikoff and Henikoff, 1992](https://doi.org/10.1073/pnas.89.22.10915)) you gave the protein aligner in @sec-biopython-align. You look it up with two amino acids, just like our `dna_scores`, so all our functions work for proteins without a single change. We only hand them a different table and a different alphabet:

In [ ]:
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")
amino_acids = "ACDEFGHIKLMNPQRSTVWY"

print(blosum62["P"]["P"], blosum62["Q"]["E"], blosum62["W"]["G"])
print(word_score("PQG", "PEG", blosum62))
print(neighborhood("PQG", 13, blosum62, amino_acids))

Unlike `dna_scores`, BLOSUM62 gives different pairs very different scores. A `Q` against an `E` scores 2, because glutamine and glutamate are similar, while a `W` against a `G` scores -2. So the neighborhood of `PQG` at $T = 13$ includes `PQG` itself, but also `PEG`, `PKG`, `PRG` and others, where the middle amino acid is one that BLOSUM62 considers a reasonable stand-in for glutamine. Neighborhoods are not just about how many letters differ, but about which ones.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Use the scores printed above to predict `word_score("PQG", "PQG", blosum62)`. You need the scores for `Q` against `Q` and `G` against `G` too, so print those first. Do all the substitution and reduction steps in your head, then run `word_score` to check.

In [ ]:
%%sandbox
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")
print(blosum62["Q"]["Q"], blosum62["G"]["G"])


#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Predict whether the neighborhood of `PQG` at $T = 11$ is larger or smaller than at $T = 13$, and then predict which word is the only other word left in the neighborhood at $T = 15$. Then check all three by printing the neighborhoods and their lengths.

In [ ]:
print(len(neighborhood("PQG", 13, blosum62, amino_acids)))

For proteins, then, the word size stays at 3, and the threshold $T$ is the setting that trades speed against sensitivity. Let us measure that too, with the same kind of experiment as for DNA: a random protein query of 150 amino acids, and a database of about 100,000 amino acids with ten relatives hidden in it, in which about half the amino acids have been replaced. This cell takes a little while to run, because our neighborhoods are computed the slow way.

In [ ]:
random.seed(1)

protein_query = random_seq(150, amino_acids)

pieces = []
protein_relatives = {}
position = 0
for _ in range(10):
    filler = random_seq(10_000, amino_acids)
    pieces.append(filler)
    position = position + len(filler)
    protein_relatives[position] = True
    pieces.append(mutate(protein_query, 0.5, amino_acids))
    position = position + len(protein_query)
protein_database = "".join(pieces)

protein_hits = {}
for T in [11, 13, 15]:
    protein_hits[T] = find_hits(build_lookup(protein_query, 3, T, blosum62, amino_acids), protein_database, 3)
    random_count = 0
    relatives_hit = set()
    for i, j in protein_hits[T]:
        if j - i in protein_relatives:
            relatives_hit.add(j - i)
        else:
            random_count = random_count + 1
    print(f"T = {T}: {len(protein_hits[T]):6} hits, {random_count:6} random, relatives found: {len(relatives_hit)} of 10")

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Raising $T$ from 11 to 15 cuts the number of random hits. By how many times? And how many relatives are still found? Then predict what happens at $T = 17$, add it to the list and run the cell again. Why does the number of relatives found hold up better here than the number of hits?

### Step 4: could it be chance? {#sec-blast-evalue}

Search a big enough database with any query and you will find something that scores well, just by chance. So BLAST must tell you how surprised you should be by a score. It does so with the *E-value*: the number of alignments with at least this score you would expect to find by chance alone in a database of random sequences of this size.

For ungapped alignments there is a formula for it, found by Samuel Karlin and Stephen Altschul ([Karlin and Altschul, 1990](https://doi.org/10.1073/pnas.87.6.2264)):

$$E = K \, m \, n \, e^{-\lambda S}$$

Here $S$ is the score, $m$ is the length of the query, $n$ is the total length of the database, and $K$ and $\lambda$ are two numbers that depend on the scoring table. For BLOSUM62 without gaps, $\lambda \approx 0.318$ and $K \approx 0.134$. Let us find a protein HSP with our functions and compute its E-value. The default values for `lam` and `K` mean you only have to give the score and the two lengths:

In [ ]:
import math

def evalue(S, m, n, lam=0.3176, K=0.134):
    return K * m * n * math.exp(-lam * S)

protein = "MSHWEKLNGCAPRTYIDVQKFGEMLW"
other = "TTGAPNSHWEKINGCAPKTFIDLRSAQPLE"
protein_hits_toy = find_hits(build_lookup(protein, 3, 13, blosum62, amino_acids), other, 3)
print(protein_hits_toy)

score, protein_part, other_part = extend(protein, other, 1, 6, 3, blosum62, 10)
print(score, protein_part, other_part)
print(evalue(score, len(protein), 200_000_000))

Read the formula as a story. The $e^{-\lambda S}$ part makes $E$ shrink very quickly as the score goes up: a high score is unlikely to happen by chance. The $m \, n$ part makes $E$ grow with the size of the search: the more places you look, the more chance hits you will find. So the same alignment gets a different E-value depending on which database you searched.

Because $K$ and $\lambda$ differ between scoring tables, raw scores cannot be compared across them. BLAST therefore also reports a *bit score*, $S' = (\lambda S - \ln K) / \ln 2$, which puts every scoring system on the same scale. With bit scores, the formula becomes simply $E = m \, n \, 2^{-S'}$. The same formula holds for DNA, with values of $K$ and $\lambda$ that belong to the DNA scores.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Predict what happens to the E-value of an alignment if the database doubles in size. And what happens if instead the score goes up by 10? Decide first, then check both with the `evalue` function. Finally, find (by trying a few values) roughly which score an alignment of our 26-letter protein needs to reach an E-value below 0.001 in a database of 200 million amino acids.

In [ ]:
%%sandbox
import math

def evalue(S, m, n, lam=0.3176, K=0.134):
    return K * m * n * math.exp(-lam * S)


## Gapped BLAST: two hits on one diagonal {#sec-blast-1997}

Seven years after the first paper, the same group published a new BLAST ([Altschul et al., 1997](https://doi.org/10.1093/nar/25.17.3389)) for proteins that was about three times faster and at least as sensitive. It changed two things.

### Asking for two clues instead of one {#sec-blast-two-hit}

The authors measured where the original BLAST spent its time. Most of it went to Step 3, extending hits, and almost all of those extensions led nowhere: a random match in a random place, extended a few letters until the X-drop rule stopped it. You saw a different kind of waste in the extension exercise: when there is a good alignment, it contains many hits on the same diagonal, and each one is extended into the same HSP.

Both observations point to the same fix. An alignment worth finding is usually long enough to contain more than one hit. A random hit, on the other hand, is rarely accompanied by another one nearby on the same diagonal. So the new BLAST only extends when it finds *two* hits on the same diagonal, that do not overlap, and that are no more than $A$ letters apart. The paper used $A = 40$. This is the *two-hit method*, and the two hits together are sometimes called a *paired seed*.

Requiring two hits means fewer extensions, but it also risks missing alignments that only have one hit. To win back that sensitivity, the authors lowered the threshold $T$ from 13 to 11. You measured above what that does: bigger neighborhoods and many more hits. But since only pairs of hits trigger an extension, the number of extensions still goes way down.

Here is our version. It sorts the hits into a dictionary by diagonal, and then looks for neighbors on each diagonal:

In [ ]:
def two_hit_seeds(hits, w, A):
    diagonals = {}
    for i, j in hits:
        d = j - i
        if d not in diagonals:
            diagonals[d] = []
        diagonals[d].append(i)

    seeds = []
    for d in diagonals:
        positions = sorted(diagonals[d])
        for k in range(1, len(positions)):
            distance = positions[k] - positions[k-1]
            if distance >= w and distance <= A:
                seeds.append((positions[k], positions[k] + d))
    return seeds

print(len(protein_hits_toy), two_hit_seeds(protein_hits_toy, 3, 40))

The hits of the small protein example became a single place to extend. The `distance >= w` makes sure the two hits do not overlap, and the `positions[k] + d` turns the query position back into a subject position, because $d = j - i$ means $j = i + d$.

Now the real comparison, on the protein database from @sec-blast-protein-words: the original one-hit BLAST with $T = 13$, against the two-hit BLAST with $T = 11$. We already have the hits for both thresholds in `protein_hits`.

In [ ]:
def count_relatives(seeds, relatives):
    hit = set()
    for i, j in seeds:
        if j - i in relatives:
            hit.add(j - i)
    return len(hit)

one_hit = protein_hits[13]
two_hit = two_hit_seeds(protein_hits[11], 3, 40)
print("one hit, T = 13: ", len(one_hit), "extensions,", count_relatives(one_hit, protein_relatives), "relatives found")
print("two hits, T = 11:", len(two_hit), "extensions,", count_relatives(two_hit, protein_relatives), "relatives found")

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

How many times fewer extensions does the two-hit version need? Does it still find the relatives? Then predict what happens if you make the relatives more distant by changing `0.5` to `0.7` in the cell that builds the protein database, and rerun both cells. Which strategy loses relatives first?

#### Exercise

[AI: Comparer](../intro/course-introduction.qmd#sec-badge-comparer){.small}

Ask the assistant for a different way to write `two_hit_seeds`, and give it the function and nothing else. Read both versions and judge which one is easier to follow and why. Then settle whether the assistant's version does the same thing by running both on `protein_hits_toy` and on `protein_hits[11]` and checking that they return the same seeds. Note in your logbook whether they did, and if not, which one you trust and how you know.

### Gaps, but only where they matter {#sec-blast-gapped}

The original BLAST found only ungapped alignments. If a related sequence had an insertion, BLAST would report two separate HSPs, one on each side of it, each with a weaker score and a worse E-value than the whole alignment deserved.

The 1997 BLAST adds gaps, but without paying for a full Smith-Waterman alignment. When an ungapped extension reaches a score high enough to be interesting, BLAST picks a point in the middle of the best part of that HSP and starts a gapped alignment from there, spreading outward in both directions. It fills in the alignment table just like Smith-Waterman does, with one difference: it uses the X-drop idea again, and stops filling in cells whose score has dropped too far below the best seen so far. In practice it only fills in a narrow band around the diagonal, which is a tiny part of the full table. Only a handful of HSPs per search ever get that far, so the gapped step costs little.

Gapped alignments come at a price in the statistics. The formula for the E-value was proven only for ungapped alignments, but it turns out to fit gapped ones too, with different values of $\lambda$ and $K$. Those values were found by aligning huge numbers of random sequences and measuring, and BLAST keeps a table of them for each combination of substitution matrix and gap scores. That is why you can only choose among a fixed menu of gap scores on the BLAST website.

The same paper also introduced PSI-BLAST, which you will meet among the variants below.

#### Exercise

[AI: Explainer](../intro/course-introduction.qmd#sec-badge-explainer){.small}

Ask the assistant to explain, in plain words, why the two-hit method made BLAST both faster and more sensitive at the same time, given that requiring two hits sounds like it should miss things. Then judge its explanation against the numbers you got from the comparison above: does it agree with what your own code showed about the number of extensions and the relatives found? Write in your logbook which parts of the explanation your own output confirmed and which parts you could not check.

## BLAST at NCBI {#sec-blast-ncbi}

Today's BLAST at NCBI, called BLAST+ ([Camacho et al., 2009](https://doi.org/10.1186/1471-2105-10-421)), is a descendant of the 1997 version. Go to [blast.ncbi.nlm.nih.gov](https://blast.ncbi.nlm.nih.gov) and look at the start page while you read this section. It offers five basic programs, and which one you need depends on two questions: is your query DNA or protein, and is the database DNA or protein?

| Program | Query | Database | What is compared |
|---|---|---|---|
| `blastn` | DNA | DNA | DNA with DNA |
| `blastp` | protein | protein | protein with protein |
| `blastx` | DNA | protein | the query translated in all six reading frames, with proteins |
| `tblastn` | protein | DNA | the protein with the database translated in all six reading frames |
| `tblastx` | DNA | DNA | the query translated in six frames, with the database translated in six frames |

Why translate at all, when you could compare DNA with DNA? Because proteins change more slowly than the DNA that codes for them. The genetic code has several codons for most amino acids, so the DNA of two related genes can drift apart while their proteins stay much the same. Comparing at the protein level finds relatives that `blastn` would miss. You saw the DNA side of this in @fig-blast-tradeoff: once a relative is only about half identical at the DNA level, exact DNA words rarely find it. The six reading frames are the three ways of splitting a DNA sequence into codons, on each of the two strands, just like you used in the ORF project (@sec-orf-project).

These are also the names you give `Blast.qblast` as its first argument, as you did with `"blastp"` in @sec-biopython-blast.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

For each of the situations below, decide which of the five programs you would use. Write down your answer and a one-sentence reason before you check it against the table.

- You have sequenced a stretch of DNA from an unknown bacterium and want to know whether it codes for a known protein.
- You have a protein from a fruit fly and want to find the genes in a newly sequenced, not yet annotated, mosquito genome that code for something similar.
- You want to know which known proteins are most similar to the FOXP2 protein.
- You have a PCR product and want to check that it matches the human gene you meant to amplify.

`tblastx` compares every one of the six frames of the query with every one of the six frames of each database sequence. How many comparisons is that for each pair of sequences? Use that to explain why `tblastx` is by far the slowest of the five.

### Choosing an algorithm {#sec-blast-algorithms}

Once you have picked nucleotide or protein BLAST on the website, the page lets you choose among variants of the algorithm under *Program Selection*. They differ in how they find hits, which is Step 1 and Step 2 above.

For nucleotide BLAST, the choice is the speed and sensitivity trade-off you measured in @sec-blast-parameters:

`megablast` is for finding sequences that are nearly identical to yours, like the same gene in another person or a closely related species. It is the fastest, because it demands an exact match of 28 bases before it extends anything ([Zhang et al., 2000](https://doi.org/10.1089/10665270050081478)). Long words give very few random hits.

`blastn` is the classic version and uses words of 11 bases. It is slower but finds more distant relatives.

`discontiguous megablast` is meant for comparing genes across species that are further apart. Its words are *spaced seeds*: it looks at, say, 11 positions spread out over a window of 16 or more bases, and ignores the positions in between ([Ma et al., 2002](https://doi.org/10.1093/bioinformatics/18.3.440)). If the ignored positions are chosen to be the third positions of codons, where mutations often do not change the amino acid, the seed still matches two genes whose DNA differs at many third positions. It gets some of the sensitivity of short words with the few random hits of long ones.

For protein BLAST, the variants differ more:

`blastp` is the standard protein search, essentially the 1997 algorithm you just built a toy version of.

`Quick BLASTP` is an accelerated version that NCBI recommends when you expect your relatives to share at least half their amino acids with your query.

`PSI-BLAST` (position-specific iterated BLAST, from the same 1997 paper) runs `blastp` and then builds a new scoring table from the significant hits it found. In that table, each position of your query gets its own scores, so a position that is always a glycine among the relatives scores very high for glycine and low for everything else, while a position where the relatives vary is more forgiving. It then searches again with that table, and repeats. Each round can find relatives too distant for the previous round to see. The danger is that a single unrelated sequence that sneaks into the table drags in its own relatives in the next round, so you must look at what is added each round.

`PHI-BLAST` only reports alignments that contain a pattern you give it, such as a known active-site motif ([Zhang et al., 1998](https://doi.org/10.1093/nar/26.17.3986)).

`DELTA-BLAST` builds a position-specific scoring table like PSI-BLAST does, but gets it from a database of known protein domains rather than from your own search ([Boratyn et al., 2012](https://doi.org/10.1186/1745-6150-7-12)). That often finds distant relatives in a single round.

### Choosing a database {#sec-blast-databases}

The database menu matters as much as the program, because the E-value depends on the size of the database, and because you can only find what is in it. For proteins, `nr` (non-redundant) holds nearly every protein sequence NCBI has, `ClusteredNR` groups very similar sequences in `nr` together so that you get one hit per group rather than a hundred near-identical ones, `refseq_protein` holds NCBI's curated reference sequences, `swissprot` holds the hand-curated UniProt/Swiss-Prot entries, and `pdb` holds proteins with a known 3D structure. For DNA, there is a large nucleotide collection along with more specialized databases, such as RefSeq transcripts and genomes. The menu changes over time as NCBI reorganizes its data, so read the description of each database on the page before you pick one.

NCBI also offers specialized tools built on BLAST, which you find further down the start page. `Primer-BLAST` designs PCR primers and checks that they only match where you want them to. `IgBLAST` analyzes antibody and T-cell receptor sequences. `SmartBLAST` gives a quick overview of the best protein matches. If you tick *Align two or more sequences* on any of the search pages, you can compare your query directly with sequences you paste in, without searching a database at all.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Go to the BLAST website, choose Protein BLAST, and search with the FOXP2 protein from @sec-biopython-features against `swissprot`. Before you press the button, look at the top hits you printed from `data/foxp2_blastp.xml` in @sec-biopython-blast and predict whether the website will give you the same top hits. Then compare. If there are differences, look at the date of each result and the settings used, and suggest why.

#### Exercise

[AI: Explainer](../intro/course-introduction.qmd#sec-badge-explainer){.small}

Ask the assistant what the E-value of a BLAST hit means, and whether an E-value of 0.01 means there is a 1% chance that the hit is a coincidence. Then settle it yourself with the formula from @sec-blast-evalue and the `evalue` function: what happens to the E-value of the very same alignment if the database were ten times smaller? Use your answer to decide whether the assistant's explanation holds up, and note in your logbook what it got right and what it got wrong.

## Biopython BLAST {#sec-biopython-blast}

An alignment compares two sequences you already have. BLAST finds the sequences in a database that are most similar to yours, and then aligns your sequence to each of them. The function `Blast.qblast` sends your sequence to NCBI's BLAST servers. You give it the BLAST program to use (`blastp` searches a protein database with a protein), the database (`swissprot` is a carefully curated protein database), and your sequence. Here we search with the FOXP2 protein you translated above.

A BLAST search at NCBI takes at least a minute, and when many people are using the servers, your search can wait in their queue for a lot longer. NCBI also asks that you do not send more than one search every ten seconds. So save the result to a file and work with the file from then on. The cell below shows how. You do not need to run it, because we have run it for you and put the result in `data/foxp2_blastp.xml`.

In [ ]:
protein

```{.python}
from Bio import Blast

Blast.email = "your.name@post.au.dk"   # use your own address

stream = Blast.qblast("blastp", "swissprot", protein, hitlist_size=20)

with open("data/foxp2_blastp.xml", "wb") as f:
    f.write(stream.read())
stream.close()
```

The `"wb"` opens the file for writing *bytes* rather than text, because that is what the stream from NCBI delivers. The result is an XML file. Open it in VS Code if you are curious, but you will be happy that you do not have to read it yourself. `Blast.read` does that:

```{.python}
from Bio import Blast

blast_record = Blast.read("data/foxp2_blastp.xml")
print(len(blast_record))
```

The result is built from objects inside objects, just like the records and features above. The `blast_record` holds one `Hit` for each database sequence that BLAST found, and you loop over it just like a list. Each `Hit` holds one or more HSPs (high-scoring segment pairs), which are the alignments between your sequence and that database sequence. Each HSP has its statistics in a dictionary called `annotations`:

```{.python}
for hit in blast_record:
    hsp = hit[0]
    print(hit.target.id, hsp.annotations["evalue"], hit.target.description[:60])
```

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Pick out the first hit and its first HSP, and print the HSP itself. Before you run it, decide what printing an HSP shows you. You have seen printed alignments above, so what would you expect it to look like?

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

`hsp.annotations["identity"]` is the number of identical positions in the alignment, and `hsp.length` is the length of the alignment. Write a loop that prints the percent identity of the first HSP of every hit, with the description of the hit. Which hits are FOXP2 from other species, and which are other genes of the same family (FOXP1, FOXP3, FOXP4)?

The first hit is human FOXP2 itself, yet it is not 100% identical to our protein. Predict why before you print its HSP and look. Hint: compare `len(protein)` with the length of the database sequence, which the printed HSP shows you, and remember that our record is *transcript variant 4*.

#### Exercise

[AI: Explainer](../intro/course-introduction.qmd#sec-badge-explainer){.small}

Ask the assistant how to print the E-value of every hit in a BLAST result with Biopython, and give it nothing else. Many answers still use an older interface, `Bio.Blast.NCBIXML`, where the E-value lives in an attribute called `hsp.expect`. Run the code it gives you on `data/foxp2_blastp.xml`. If it fails, read the error and say which attribute or function does not exist in the Biopython you have installed. Settle it against the BLAST chapter of the [Biopython Tutorial](https://biopython.org/docs/latest/Tutorial/chapter_blast.html), and write in your logbook what the assistant claimed and what actually ran.